# Gemma 3 폴백 검증 노트북

Gemma 4가 막힐 경우(또는 torch 2.5.1 강제 시)를 대비한 **폴백 모델 Gemma 3** 동작 확인용.
**SSH 박스와 Colab T4 양쪽에서** 로드·생성·RAG스타일 grounding이 되는지 본다.

- 모델: `google/gemma-3-12b-it` (4bit NF4, float16) — 더 가벼우면 `gemma-3-4b-it`
- Gemma 3는 transformers 5.x(현재) + torch 2.11 에서 동작 (torch 2.5.1 강제 시엔 transformers 4.49로 내려야 함)
- **gemma-3는 gated 모델** → HF 토큰 필요 (Colab은 아래 로그인 셀 사용)

## 실행 순서
1. [환경 확인] 셀 — torch/transformers/GPU 확인
2. (Colab만) [HF 로그인] 셀
3. [모델 로드] 셀
4. [생성 테스트] + [RAG 스타일 테스트] 셀

## [환경 확인]

In [ ]:
# (Colab) 필요 시 설치 — torch는 그대로 두고 라이브러리만 최신화
# !pip install -q -U transformers accelerate bitsandbytes

import torch, transformers
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("transformers:", transformers.__version__)

## [HF 로그인] (Colab만 — gemma-3는 gated)
SSH 박스는 이미 인증돼 있으면 건너뛰어도 됨.

In [ ]:
# from huggingface_hub import login
# login()  # 토큰 입력창이 뜸 (또는 os.environ['HF_TOKEN']='hf_...' 설정)

## [모델 로드] Gemma 3 12B (4bit NF4)

In [ ]:
import os
os.environ.setdefault("HF_HOME", "/content/hf_cache")  # Colab 로컬 캐시 (Drive 쿼터 회피)

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-12b-it"   # 더 가볍게: "google/gemma-3-4b-it"
COMPUTE_DTYPE = torch.float16        # T4/RTX8000(Turing)은 bf16 텐서코어 없음 → float16

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print("[1/2] 프로세서 로드")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("[2/2] 모델 로드 (4bit NF4)")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)
model.eval()
print(f"\n\u2705 \ub85c\ub4dc \uc644\ub8cc \u2014 VRAM {torch.cuda.memory_reserved()/1e9:.2f} GB")

## [생성 테스트] 한국어 한 문장

In [ ]:
def gen(text, max_new_tokens=256):
    """단일 user 메시지로 생성 (Gemma 3는 system role 미지원 → user에 합침)."""
    messages = [{"role": "user", "content": [{"type": "text", "text": text}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    in_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, repetition_penalty=1.3,
        )
    return processor.decode(out[0][in_len:], skip_special_tokens=True).strip()

print(gen("\ucda9\ub0a8\ub300\ud559\uad50\uc5d0 \ub300\ud574 \ud55c \ubb38\uc7a5\uc73c\ub85c \uc124\uba85\ud574\uc918."))

## [RAG 스타일 테스트] 시스템 지시 + 하드코딩 컨텍스트 + 질문
벡터DB 없이도 grounding/한국어 품질 감을 본다. (실제 파이프라인은 src/ 사용)

In [ ]:
SYSTEM = (
    "\ub108\ub294 \ucda9\ub0a8\ub300\ud559\uad50 \ud559\ub0b4 \uc815\ubcf4\ub97c \uc548\ub0b4\ud558\ub294 \uce5c\uc808\ud55c AI \ucc57\ubd07\uc774\uc57c. "
    "\ucc38\uace0 \uc790\ub8cc\uc5d0 \uc788\ub294 \ub0b4\uc6a9\ub9cc\uc73c\ub85c \ub2f5\ud558\uace0, \uc5c6\uc73c\uba74 '\ud655\uc778\ub418\uc9c0 \uc54a\uc558\uc5b4\uc694'\ub77c\uace0 \uc194\uc9c1\ud788 \ub2f5\ud574."
)
CONTEXT = (
    "\ucef4\ud4e8\ud130\uc735\ud569\ud559\ubd80 \uc878\uc5c5\uc694\uac74: \ud559\uc0ac\ud559\uc704 \uc878\uc5c5\uc5d0 \ud544\uc694\ud55c \ucd1d \ud559\uc810\uc740 130\ud559\uc810\uc774\ub2e4. "
    "\uad50\uc591\uacfc\ubaa9\uc740 \ucd5c\ub300 48\ud559\uc810\uae4c\uc9c0 \uc778\uc815(2025\ud559\ub144\ub3c4 \uae30\uc900). \ud504\ub85c\uc81d\ud2b8 \uad50\uacfc\ubaa9 \ucd5c\uc18c 3\uac1c \uc774\uc218."
)
Q = "\ucef4\ud4e8\ud130\uc735\ud569\ud559\ubd80 \uc878\uc5c5\ud558\ub824\uba74 \uba87 \ud559\uc810 \ub4e4\uc5b4\uc57c \ud574?"

prompt = f"{SYSTEM}\n\n\ucc38\uace0 \uc790\ub8cc:\n{CONTEXT}\n\n\uc9c8\ubb38: {Q}"
print(gen(prompt, max_new_tokens=300))